# multi_heatmap - 02 - Modelo, entrenamiento y validacion

Variante `universal_refs`: nucleo lineal instantaneo compartido entre configuraciones REALES (19/39/64 + externas BIDS), **cabeza temporal de residuo** sobre ventanas centradas offline y restricciones fisicas (modos l<=1, campo de superficie, consistencia cross-config).

**Uso:** Runtime -> GPU (T4). Ejecutar antes `01_exploracion_datos.ipynb`.

## 0 - Entorno

In [ ]:
# ---- 0 - Entorno (Colab o local) --------------------------------------
# Colab: Runtime > Change runtime type > GPU (T4). Rama: explore/multi-heatmap.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/sonoAESS/universal-eeg-transformer.git"
BRANCH   = "explore/multi-heatmap"

IN_COLAB = "google.colab" in sys.modules or "/content" in os.getcwd()

if IN_COLAB:
    ROOT = Path("/content/universal-eeg-transformer")
    if not ROOT.exists():
        !git clone -b {BRANCH} {REPO_URL} {ROOT}
    %pip install -q mne pyyaml pandas matplotlib scikit-learn requests scipy
else:
    ROOT = Path.cwd()
    while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print(f"ROOT = {ROOT}\nColab = {IN_COLAB}")

In [ ]:
# ---- 0b - Persistencia en Google Drive (opcional) ---------------------
USE_DRIVE = IN_COLAB

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/universal_eeg_cache")
    CACHE.mkdir(exist_ok=True)
    for link in ("data/processed", "runs"):
        target = CACHE / link.split("/")[-1]
        target.mkdir(parents=True, exist_ok=True)
        dest = ROOT / link
        if not dest.exists():
            dest.symlink_to(target)
    print("Cache y runs enlazados a:", CACHE)
else:
    print("Modo local: cache en ./data/processed y ./runs")

## 1 - Experimento (con flag de ablation)

In [ ]:
# ---- 1 - Experimento ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)
plt.rcParams["figure.dpi"] = 110

CONFIG = ROOT / "config/universal_refs.yaml"
FORCE    = False    # True = reentrenar desde cero
ABLATION = False    # True = entrenar SIN cabeza temporal (instantaneo puro)

from eeg_transform.nb import (
    evaluate_multiconfig, evaluate_multiconfig_surface,
    load_experiment, multiconfig_data, plot_multiconfig_bars,
    plot_multiconfig_heatmap, plot_multiconfig_scalps,
    plot_multiconfig_surface, plot_training, train_variant,
)
from eeg_transform.training.trainer import build_multiconfig_model

if ABLATION:
    # ablation: mismo pipeline con la cabeza desactivada (run aparte)
    import yaml as _y
    from eeg_transform.config import save_config
    raw = _y.safe_load(CONFIG.read_text())
    raw["model"]["temporal_window"] = 0
    raw["training"]["run_dir"] = "runs/universal_refs_ablation"
    raw["training"]["log_file"] = "runs/universal_refs_ablation/train.log"
    CONFIG = ROOT / "config/_universal_refs_ablation.yaml"
    save_config(raw, CONFIG)

cfg, ds = load_experiment(CONFIG)
data = multiconfig_data(cfg, ds)
print(ds.summary())
print("variante:", cfg.model.variant,
      "| ventana:", cfg.model.temporal_window or "OFF (instantaneo)")

## 2 - Arquitectura

In [ ]:

from eeg_transform.models.multi_heatmap import MultiHeatmapAutoencoder

In [ ]:
# ---- 2 - Arquitectura hibrida ------------------------------------------
model = build_multiconfig_model(cfg, data)

n_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
has_head = getattr(cfg.model, "temporal_window", 0) > 0 and hasattr(model, "head")
head_params = sum(int(np.prod(v.shape))
                  for v in model.head.trainable_variables) if has_head else 0
print(f"{cfg.model.variant} | latente {model.n_canonical} | "
      f"params {n_params:,} (cabeza temporal: {head_params:,})")
for lbl in model.configs:
    print(f"  {lbl:18s} C_s={model.projections[lbl].shape[0]:>3d}")

if has_head:
    # comprobacion fisica clave: cabeza a cero => salida == modelo lineal
    c0 = model.projections[model.configs[0]]
    rngx = np.random.default_rng(0).normal(
        size=(2, cfg.model.temporal_window, c0.shape[0])).astype("float32")
    lbl0 = model.configs[0]
    out_dyn = dict(model._predict_cfg(rngx, lbl0, "unipolar"))
    out_lin = MultiHeatmapAutoencoder._predict_cfg(model, rngx, lbl0, "unipolar")
    ok = all(np.allclose(out_dyn[k].numpy(), out_lin[k].numpy(), atol=1e-5)
             for k in out_dyn)
    print("arranque == instantaneo:", ok)

## 3 - Entrenamiento ventaneado

In [ ]:
# ---- 3 - Entrenamiento -------------------------------------------------
# Con temporal_window > 0 el trainer consume VENTANAS automaticamente
# (build_multiconfig_windowed); early stopping y checkpoint como siempre.
model, history = train_variant(cfg, ds, force=FORCE)
run_dir = Path(cfg.training.run_dir)
print("run_dir:", run_dir)

## 4 - Evaluacion

In [ ]:
# ---- 4 - Evaluacion: 49 rutas por configuracion ------------------------
metrics_df, summary = evaluate_multiconfig(cfg, ds, model, data)
print("=== RESUMEN (RMSE uV): modelo vs linea base analitica ===")
print(summary.round(3).to_string(index=False))

focus = metrics_df[(metrics_df.destino.isin(["laplacian", "rest"]))
                   & (metrics_df.origen != metrics_df.destino)]
if len(focus):
    print("\n=== CRUZADAS hacia rest/laplacian ===")
    print(focus.groupby(["origen", "destino"])[["rmse", "rmse_ana"]]
          .mean().round(4).to_string())

## 5 - Superficie

In [ ]:
# ---- 5 - Campo de superficie -------------------------------------------
surface_df, surf_summary = evaluate_multiconfig_surface(model, data)
print(surf_summary.round(3).to_string(index=False))

## 6 - Espectral

In [ ]:
# ---- 6 - Diagnostico espectral por bandas ------------------------------
# Donde vive el error de la ruta unipolar->laplacian: si la cabeza dinamica
# aporta, el residuo se concentra fuera de las bandas oscilatorias.
from eeg_transform.evaluation.metrics import spectral_band_table

lbl = "canonical" if "canonical" in data.order else data.order[0]
mc = data[lbl]
X = mc.refs["test"]["unipolar"]
Y_true = mc.refs["test"]["laplacian"]
T_w = cfg.model.temporal_window or 32
pad = T_w // 2
step = max(1, T_w // 2)

preds = np.zeros_like(Y_true)
counts = np.zeros(len(Y_true))
for start in range(0, max(1, len(X) - T_w + 1), step):
    win = X[start : start + T_w][None].astype("float32")
    outs = model(win, source="unipolar")
    center = slice(start + pad, min(start + pad + step, len(Y_true)))
    off = center.start - start
    n = center.stop - center.start
    preds[center] = outs["laplacian"].numpy()[0, off : off + n]
    counts[center] += 1
valid = counts > 0
tabla = spectral_band_table(Y_true[valid], preds[valid], sfreq=160.0)
print(tabla.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(tabla["banda"], tabla["rmse"])
ax.set_title(f"RMSE por banda - {lbl}: unipolar->laplacian")
ax.set_ylabel("RMSE (V/m^2)")
plt.tight_layout(); plt.show()

## 7 - Figuras

In [ ]:
# ---- 7 - Figuras -------------------------------------------------------
plot_training(run_dir / "history.csv")
plt.show()

plot_multiconfig_heatmap(metrics_df,
                         title=f"RMSE multi-config (uV, log10) - {cfg.model.variant}")
plt.show()

plot_multiconfig_bars(metrics_df,
                      title=f"RMSE/ve vs analisis - {cfg.model.variant}")
plt.show()

plot_multiconfig_surface(surface_df,
                         title=f"Campo de superficie - {cfg.model.variant}")
plt.show()

for label, fig in plot_multiconfig_scalps(cfg, model, data=data).items():
    print(f"--- {label} ---")
    plt.show()

## 8 - Guardar

In [ ]:
# ---- 8 - Guardar artefactos --------------------------------------------
metrics_df.to_csv(run_dir / "metrics_test.csv", index=False)
surface_df.to_csv(run_dir / "metrics_surface_test.csv", index=False)
summary.to_csv(run_dir / "summary_test.csv", index=False)
model.save_weights(run_dir / "best.weights.h5")
print("Artefactos en:", run_dir)

## Interpretacion y ablation

* **Diagonal vs cruzada** intra-configuracion frente a `*_ana`: el modelo
  lineal debe superar a la pseudo-inversa rigida; las rutas hacia
  `laplacian`/`rest` desde cascos escasos son el terreno de la cabeza
  temporal.
* **Tabla espectral**: error concentrado en gamma/beta sugiere contenido
  de alta frecuencia no capturado; si la cabeza dinamica ayuda, la
  version instantanea (ablation) mostrara peor `ve` justo en esas bandas.
* **Ablation**: reejecutar con `ABLATION = True` entrena el mismo pipeline
  sin cabeza (`temporal_window = 0`, run separado) y permite comparar
  rutas/bandas directamente.

Documentacion: `docs/guia_conceptual.md`, AGENTS.md de la rama.